In [ ]:
# Input: one MetaPhlAn *.txt profile per sample in metaphlan_profiles, or a merged abundance table with index followed by species columns.
# Input: one saved .joblib model dictionary per metabolite in models (or Models); target, model, and feature_columns are read from each file.
# Input paths are configured below; abundances must be numeric and non-negative.

from pathlib import Path
PROJECT_ROOT = Path.cwd()
NOTEBOOK_DIR = PROJECT_ROOT
RAW_METAPHLAN_DIR = PROJECT_ROOT / "metaphlan_profiles"
SPECIES_EACH_SAMPLE_DIR = PROJECT_ROOT / "species_tables"
MERGED_SPECIES_OUTPUT = PROJECT_ROOT / "merged_species_abundance.txt"
PREDICTION_INPUT_FILE = MERGED_SPECIES_OUTPUT
MODEL_DIR = PROJECT_ROOT / "models"
if not MODEL_DIR.is_dir():
    MODEL_DIR = PROJECT_ROOT / "Models"
PREDICTION_OUTPUT_DIR = PROJECT_ROOT / "prediction_results"
RUN_EXTRACT_SPECIES = True
RUN_MERGE_SPECIES = True
RUN_PREDICTION = True
print("Paths and run switches configured.")


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import time

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display

TARGETS = ["TA", "IAM", "ILA", "I3C", "IE", "IAA", "IA", "IPA", "INDOLE"]

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".txt", ".tsv"}:
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path}")

def write_table(frame: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        frame.to_csv(path, index=False, encoding="utf-8-sig")
    elif suffix in {".txt", ".tsv"}:
        frame.to_csv(path, sep="\t", index=False)
    elif suffix in {".xlsx", ".xls"}:
        frame.to_excel(path, index=False)
    else:
        raise ValueError(f"Unsupported table format: {path}")


In [ ]:
def extract_species_tables(
    input_dir: Path,
    output_dir: Path,
) -> pd.DataFrame:
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"MetaPhlAn input directory not found: {input_dir}")

    files = sorted(input_dir.glob("*.txt"))
    if not files:
        raise FileNotFoundError(f"No *.txt files found in: {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []
    start_time = time.time()

    for i, file in enumerate(files, 1):
        source_stem = file.stem
        sample_name = source_stem.rstrip("_") or source_stem
        rows = []

        with file.open("r", encoding="utf-8", errors="ignore") as source:
            for line in source:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue

                parts = line.split("\t")
                if not parts or parts[0] in {"clade_name", "ID", "name"}:
                    continue

                clade = parts[0]
                if "s__" not in clade or "t__" in clade:
                    continue

                abundance = None
                if len(parts) >= 3:
                    try:
                        abundance = float(parts[2])
                    except ValueError:
                        pass
                if abundance is None and len(parts) >= 2:
                    try:
                        abundance = float(parts[1])
                    except ValueError:
                        pass
                if abundance is None:
                    continue

                species_name = clade.split("|")[-1]
                if species_name.startswith("s__"):
                    species_name = species_name[3:]
                rows.append((species_name, abundance))

        if not rows:
            print(f"Warning: {file.name} contains no species; skipped.")
            summary_rows.append({"sample": sample_name, "n_species": 0, "status": "skipped"})
            continue

        frame = pd.DataFrame(rows, columns=["species", sample_name])
        frame = frame.groupby("species", as_index=False)[sample_name].sum()
                                                      
        output_file = output_dir / f"{source_stem}_species.txt"
        frame.to_csv(output_file, sep="\t", index=False)
        summary_rows.append({"sample": sample_name, "n_species": len(frame), "status": "saved"})

        if i % 1000 == 0 or i == len(files):
            print(f"[{i}/{len(files)}] processed", flush=True)

    summary = pd.DataFrame(summary_rows)
    n_saved = int((summary["status"] == "saved").sum()) if not summary.empty else 0
    if n_saved == 0:
        raise RuntimeError("No valid species were extracted from any input file.")

    print(f"Completed: read {len(files)} files and saved {n_saved} samples.")
    print(f"Output directory: {output_dir}")
    print(f"Elapsed: {time.time() - start_time:.2f} seconds")
    return summary


In [ ]:
if RUN_EXTRACT_SPECIES:
    species_extraction_summary = extract_species_tables(
        RAW_METAPHLAN_DIR,
        SPECIES_EACH_SAMPLE_DIR,
    )
    display(species_extraction_summary.head())
else:
    print("Step 1 skipped. Set RUN_EXTRACT_SPECIES to True to run.")


In [ ]:
def merge_species_tables(
    input_dir: Path,
    output_file: Path,
) -> pd.DataFrame:
    input_dir = Path(input_dir)
    output_file = Path(output_file)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Species table directory not found: {input_dir}")

    files = sorted(input_dir.glob("*_species.txt"))
    if not files:
        raise FileNotFoundError(f"No *_species.txt files found in: {input_dir}")

    all_series = []
    sample_names = set()
    start_time = time.time()

    for i, file in enumerate(files, 1):
        frame = pd.read_csv(file, sep="\t")
        if frame.shape[1] < 2:
            print(f"Warning: {file.name} has fewer than two columns; skipped.")
            continue

        species_col, abundance_col = frame.columns[:2]
        frame = frame[[species_col, abundance_col]].copy()
        frame = frame.rename(columns={species_col: "species"})
        sample_name = str(abundance_col)
        if sample_name == "relative_abundance":
            sample_name = file.name.removesuffix("_species.txt")
        sample_name = sample_name.rstrip("_") or sample_name
        if re.fullmatch(r"\d+-\d+", sample_name):
            sample_name = "S" + sample_name
        if sample_name in sample_names:
            raise ValueError(f"Duplicate sample name: {sample_name}")

        frame[abundance_col] = pd.to_numeric(frame[abundance_col], errors="coerce")
        frame = frame.dropna(subset=[abundance_col])
        if frame.empty:
            print(f"Warning: {file.name} has no valid abundances; skipped.")
            continue

        series = frame.groupby("species")[abundance_col].sum()
        series.name = sample_name
        all_series.append(series)
        sample_names.add(sample_name)

        if i % 1000 == 0 or i == len(files):
            print(f"[{i}/{len(files)}] read", flush=True)

    if not all_series:
        raise RuntimeError("No valid species abundance tables to merge.")

    species_by_sample = pd.concat(all_series, axis=1).fillna(0)
    prediction_ready = species_by_sample.T
    prediction_ready.index.name = "index"
    prediction_ready.columns.name = None
    prediction_ready = prediction_ready.reset_index()

    duplicate_columns = prediction_ready.columns[prediction_ready.columns.duplicated()].tolist()
    if duplicate_columns:
        raise ValueError(f"Duplicate species columns in merged abundance table: {duplicate_columns[:20]}")
    if prediction_ready["index"].duplicated().any():
        raise ValueError("Duplicate sample IDs in merged abundance table.")

    write_table(prediction_ready, output_file)
    print(f"Completed: {len(prediction_ready)} samples and {prediction_ready.shape[1] - 1} species.")
    print(f"Output file: {output_file}")
    print(f"Elapsed: {time.time() - start_time:.2f} seconds")
    return prediction_ready


In [ ]:
if RUN_MERGE_SPECIES:
    merged_species_abundance = merge_species_tables(
        SPECIES_EACH_SAMPLE_DIR,
        MERGED_SPECIES_OUTPUT,
    )
    display(merged_species_abundance.iloc[:5, :8])
else:
    print("Step 2 skipped. Set RUN_MERGE_SPECIES to True to run.")


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def require_unique_columns(frame: pd.DataFrame, label: str) -> None:
    duplicates = frame.columns[frame.columns.duplicated()].tolist()
    if duplicates:
        raise ValueError(f"{label} has duplicate columns: {duplicates[:20]}")

def numeric_abundance(frame: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    numeric = frame.apply(pd.to_numeric, errors="coerce")
    invalid = numeric.isna() & frame.notna()
    if invalid.any().any():
        locations = np.argwhere(invalid.to_numpy())[:20]
        examples = [
            {
                "row_index": int(row),
                "column": str(frame.columns[int(column)]),
                "value": str(frame.iat[int(row), int(column)]),
            }
            for row, column in locations
        ]
        raise ValueError(f"Non-numeric abundance values: {examples}")

    negative_count = int((numeric < 0).sum().sum())
    if negative_count:
        raise ValueError(f"Prediction table contains {negative_count} negative values")
    missing_cells_filled_zero = int(numeric.isna().sum().sum())
    return numeric.fillna(0.0), missing_cells_filled_zero

def load_and_validate_artifacts(
    model_dir: Path,
) -> tuple[dict[str, dict[str, object]], list[str], pd.DataFrame]:
    model_paths = sorted(Path(model_dir).glob("*.joblib"))
    if not model_paths:
        raise FileNotFoundError(f"No .joblib model files found in: {model_dir}")
    loaded_models = {}
    for artifact_path in model_paths:
        artifact = joblib.load(artifact_path)
        if not isinstance(artifact, dict):
            raise ValueError(f"Expected a model dictionary: {artifact_path}")
        target = artifact.get("target")
        if target not in TARGETS:
            raise ValueError(f"Unexpected model target {target!r}: {artifact_path}")
        if target in loaded_models:
            raise ValueError(f"Multiple saved models found for target: {target}")
        loaded_models[target] = (artifact_path, artifact)
    missing_targets = set(TARGETS) - set(loaded_models)
    if missing_targets:
        raise ValueError(f"Missing saved models for: {sorted(missing_targets)}")
    artifacts = {}
    common_features = None
    usage_rows = []

    for target in TARGETS:
        artifact_path, artifact = loaded_models[target]
        expected_model_name = artifact.get("model_name")
        if not expected_model_name or not callable(getattr(artifact.get("model"), "predict", None)):
            raise ValueError(f"Invalid saved model: {artifact_path}")
        if artifact.get("x_transform") != "log2(x + 1)":
            raise ValueError(f"Unexpected X transformation: {artifact_path}")
        if artifact.get("y_transform") != "log2(y + 1)":
            raise ValueError(f"Unexpected y transformation: {artifact_path}")

        features = list(artifact.get("feature_columns", []))
        if not features or len(features) != len(set(features)):
            raise ValueError(f"Invalid model feature list: {artifact_path}")
        if common_features is None:
            common_features = features
        elif features != common_features:
            raise ValueError("The nine models do not share identical features in the same order.")

        artifacts[target] = artifact
        usage_rows.append({
            "Target": target,
            "Model": expected_model_name,
            "Training_N": int(artifact["training_n"]),
            "N_Features": len(features),
            "Model_File": str(artifact_path),
            "Model_File_SHA256": sha256(artifact_path),
        })

    return artifacts, common_features, pd.DataFrame(usage_rows)

def predict_metabolites(
    prediction_input_file: Path,
    model_dir: Path,
    output_dir: Path,
) -> dict[str, Path]:
    prediction_input_file = Path(prediction_input_file).resolve()
    model_dir = Path(model_dir).resolve()
    output_dir = Path(output_dir).resolve()

    if not prediction_input_file.is_file():
        raise FileNotFoundError(f"Prediction input file not found: {prediction_input_file}")
    if not model_dir.is_dir():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")

    artifacts, feature_columns, model_usage = load_and_validate_artifacts(
        model_dir
    )
    prediction_input = read_table(prediction_input_file)
    if prediction_input.empty:
        raise ValueError("Prediction input table contains no samples.")
    require_unique_columns(prediction_input, "Prediction input table")
    if prediction_input.columns[0] != "index":
        raise ValueError("The first prediction input column must be named index.")
    if prediction_input["index"].isna().any():
        raise ValueError("Sample IDs contain missing values.")
    if prediction_input["index"].duplicated().any():
        duplicates = prediction_input.loc[
            prediction_input["index"].duplicated(keep=False), "index"
        ].astype(str)
        raise ValueError(f"Duplicate sample IDs: {duplicates.head(20).tolist()}")

    species_columns = [column for column in prediction_input.columns if column != "index"]
    if not species_columns:
        raise ValueError("Prediction input table contains no species columns.")
    raw_abundance, missing_cells_filled_zero = numeric_abundance(
        prediction_input[species_columns]
    )

    shared_features = [feature for feature in feature_columns if feature in raw_abundance.columns]
    missing_features = [feature for feature in feature_columns if feature not in raw_abundance.columns]
    extra_features = [feature for feature in species_columns if feature not in feature_columns]
    if not shared_features:
        raise ValueError(
            "No species features match the training models. "
            "Check matrix orientation: one sample per row, index as the first column, and species names as remaining column headers."
        )
    aligned_abundance = raw_abundance.reindex(columns=feature_columns, fill_value=0.0)
    if aligned_abundance.columns.tolist() != feature_columns:
        raise RuntimeError("Aligned features do not retain the training order.")
    if missing_features and not aligned_abundance[missing_features].eq(0.0).all().all():
        raise RuntimeError("Missing training features were not correctly filled with zeros.")

    x_log2 = np.log2(aligned_abundance.to_numpy(dtype=float) + 1.0)
    sample_ids = prediction_input["index"].astype(str).reset_index(drop=True)
    predictions_log2 = pd.DataFrame({"index": sample_ids})
    predictions_original = pd.DataFrame({"index": sample_ids})
    prediction_summary_rows = []
    clipped_total = 0

    for target in TARGETS:
        artifact = artifacts[target]
        predicted_log2 = np.asarray(artifact["model"].predict(x_log2), dtype=float).reshape(-1)
        predicted_unclipped = np.power(2.0, predicted_log2) - 1.0
        clipped_count = int((predicted_unclipped < 0.0).sum())
        clipped_total += clipped_count
        predicted_original = np.maximum(predicted_unclipped, 0.0)
        predictions_log2[target] = predicted_log2
        predictions_original[target] = predicted_original
        prediction_summary_rows.append({
            "Target": target,
            "Model": artifact["model_name"],
            "Prediction_Min_Log2": float(predicted_log2.min()),
            "Prediction_Max_Log2": float(predicted_log2.max()),
            "Prediction_Min_Original": float(predicted_original.min()),
            "Prediction_Max_Original": float(predicted_original.max()),
            "N_Inverse_Predictions_Clipped_To_Zero": clipped_count,
        })

    output_dir.mkdir(parents=True, exist_ok=True)
    stem = prediction_input_file.stem
    output_paths = {
        "original_predictions": output_dir / f"{stem}_predictions_original.csv",
        "log2_predictions": output_dir / f"{stem}_predictions_log2.csv",
        "feature_alignment": output_dir / f"{stem}_feature_alignment.csv",
        "ignored_features": output_dir / f"{stem}_ignored_features.csv",
        "model_usage": output_dir / f"{stem}_model_usage.csv",
        "prediction_summary": output_dir / f"{stem}_prediction_summary.csv",
        "metadata": output_dir / f"{stem}_run_metadata.json",
    }

    predictions_original.to_csv(output_paths["original_predictions"], index=False, encoding="utf-8-sig")
    predictions_log2.to_csv(output_paths["log2_predictions"], index=False, encoding="utf-8-sig")
    pd.DataFrame({
        "Training_Feature_Order": np.arange(1, len(feature_columns) + 1),
        "Feature": feature_columns,
        "Present_In_Prediction_Input": [feature in raw_abundance.columns for feature in feature_columns],
        "Alignment_Action": [
            "used_input_value" if feature in raw_abundance.columns else "filled_zero"
            for feature in feature_columns
        ],
    }).to_csv(output_paths["feature_alignment"], index=False, encoding="utf-8-sig")
    pd.DataFrame({"Ignored_Extra_Feature": extra_features}).to_csv(
        output_paths["ignored_features"], index=False, encoding="utf-8-sig"
    )
    model_usage.to_csv(output_paths["model_usage"], index=False, encoding="utf-8-sig")
    pd.DataFrame(prediction_summary_rows).to_csv(
        output_paths["prediction_summary"], index=False, encoding="utf-8-sig"
    )

    metadata = {
        "prediction_input_file": str(prediction_input_file),
        "prediction_input_file_sha256": sha256(prediction_input_file),
        "model_directory": str(model_dir),
        "n_samples": len(prediction_input),
        "sample_id_column": "index",
        "n_input_species_columns": len(species_columns),
        "n_training_features": len(feature_columns),
        "n_shared_training_features": len(shared_features),
        "n_missing_training_features_filled_zero": len(missing_features),
        "missing_training_features_filled_zero": missing_features,
        "n_extra_input_features_ignored": len(extra_features),
        "n_missing_input_cells_filled_zero": missing_cells_filled_zero,
        "targets": TARGETS,
        "x_processing": "numeric conversion; missing cells and absent training features -> 0; log2(x + 1)",
        "prediction_original_scale": "max(2**prediction_log2 - 1, 0)",
        "n_inverse_predictions_clipped_to_zero": clipped_total,
        "software": {
            "python": platform.python_version(),
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "scikit_learn": sklearn.__version__,
            "xgboost": xgboost.__version__,
            "joblib": joblib.__version__,
        },
    }
    output_paths["metadata"].write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    expected_columns = ["index"] + TARGETS
    for label in ("original_predictions", "log2_predictions"):
        output = pd.read_csv(output_paths[label])
        if output.columns.tolist() != expected_columns:
            raise RuntimeError(f"{label} output column validation failed.")
        if len(output) != len(prediction_input):
            raise RuntimeError(f"{label} output row count validation failed.")
        if output["index"].astype(str).tolist() != sample_ids.tolist():
            raise RuntimeError(f"{label} sample order validation failed.")
        if not np.isfinite(output[TARGETS].to_numpy(dtype=float)).all():
            raise RuntimeError(f"{label} contains non-finite predictions.")
    if (predictions_original[TARGETS] < 0).any().any():
        raise RuntimeError("Original-scale predictions contain negative values.")

    run_summary = {
        "samples": len(prediction_input),
        "input_species_columns": len(species_columns),
        "training_features": len(feature_columns),
        "shared_features": len(shared_features),
        "missing_features_filled_zero": len(missing_features),
        "extra_features_ignored": len(extra_features),
        "missing_input_cells_filled_zero": missing_cells_filled_zero,
        "inverse_predictions_clipped_to_zero": clipped_total,
        "original_predictions": str(output_paths["original_predictions"]),
    }
    print(json.dumps(run_summary, ensure_ascii=False, indent=2))
    return output_paths


In [ ]:
if RUN_PREDICTION:
    prediction_output_paths = predict_metabolites(
        PREDICTION_INPUT_FILE,
        MODEL_DIR,
        PREDICTION_OUTPUT_DIR,
    )
    display(pd.read_csv(prediction_output_paths["original_predictions"]).head())
else:
    print("Step 3 skipped. Set RUN_PREDICTION to True to run.")
